# Notebook 20: LangGraph - Advanced Control Flow

**🎯 Goal:** To master advanced control flow techniques in LangGraph, including managing cycles safely, executing nodes in parallel, and making workflows resilient with persistence and checkpointing.

## 🧩 Concept Overview

Beyond simple conditional routing, LangGraph provides tools for more complex and robust workflow orchestration.

- **Cycles & Loops:** We've seen simple loops, but we'll focus on making them safe to prevent infinite execution.
- **Parallel Execution:** How to fan-out to run multiple tasks at once.
- **Persistence & Checkpoints:** How to save the state of a long-running workflow so it can be resumed later. This is critical for production systems.

### 5.1. Cycles and Loops with Safe Exits

An agent that gets stuck in a loop is a common bug. To prevent this, you should always design your loops with a clear **exit condition**.

Common strategies include:
1.  **Max Step Counter:** Stop after a certain number of iterations.
2.  **Confidence Score:** Have the agent rate its own work and exit when the score is high enough.
3.  **Tool Validation:** Exit after a tool call succeeds or fails in a specific way.

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END

class SafeLoopState(TypedDict):
    iteration: int
    max_iterations: int
    work_quality: int

def work_node(state: SafeLoopState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"--- Iteration {iteration} ---")
    # Simulate work being done and quality improving
    work_quality = state.get("work_quality", 50) + 10
    return {"iteration": iteration, "work_quality": work_quality}

def loop_condition(state: SafeLoopState) -> str:
    if state["iteration"] >= state["max_iterations"]:
        print("--- Max iterations reached. Exiting. ---")
        return "end"
    if state["work_quality"] >= 80:
        print("--- Work quality sufficient. Exiting. ---")
        return "end"
    print("--- Work quality not yet sufficient. Continuing. ---")
    return "continue"

workflow = StateGraph(SafeLoopState)
workflow.add_node("work", work_node)
workflow.set_entry_point("work")
workflow.add_conditional_edges("work", loop_condition, {"continue": "work", "end": END})

app = workflow.compile()
app.invoke({"max_iterations": 5, "work_quality": 0, "iteration": 0})

### 5.2. Parallel Execution

Sometimes you want to perform multiple tasks at the same time (e.g., search two different sources for information). You can achieve this by having a node return a list of states, which LangGraph will then run in parallel.

**Note:** True parallelism in Python often requires `asyncio`. The following is a conceptual demonstration. In LangGraph, you can achieve this by having a node that returns a list of outputs, which are then processed by subsequent nodes.

In [ ]:
# This is a conceptual example. LangGraph's parallelism is more implicit.
# You can have one node that calls multiple tools/LLMs asynchronously.
import asyncio

class ParallelState(TypedDict):
    results: list

async def search_source_one():
    await asyncio.sleep(1)
    return "Result from Source 1"

async def search_source_two():
    await asyncio.sleep(2)
    return "Result from Source 2"

async def fan_out_node(state: ParallelState) -> dict:
    print("--- Fanning Out ---")
    # Run tasks in parallel
    parallel_results = await asyncio.gather(search_source_one(), search_source_two())
    return {"results": parallel_results}

async def fan_in_node(state: ParallelState) -> dict:
    print("--- Fanning In ---")
    return {"results": ", ".join(state['results'])}

# You would then structure your graph with these async nodes
print("Parallel execution is typically managed within a single async node.")

### 5.3. Persistence & Checkpoints

For any agent that might run for a long time, persistence is essential. If the agent crashes, you want to be able to resume it from where it left off.

LangGraph's `checkpointer` API handles this automatically. You provide a storage backend (like in-memory, SQLite, Redis, or Postgres), and it will save the state at every step.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver is the simplest checkpointer, for in-memory storage.
# For production, you'd use something like `PostgresSaver`.
memory = MemorySaver()

# Re-using the safe loop graph from before
app_with_checkpoint = workflow.compile(checkpointer=memory)

# A 'thread_id' is used to identify a specific run of the graph
thread_id = "my-long-running-thread-1"
config = {"configurable": {"thread_id": thread_id}}

# Run the graph - it will automatically save its state
app_with_checkpoint.invoke({"max_iterations": 5, "work_quality": 0, "iteration": 0}, config)

# You can inspect the saved state
saved_state = app_with_checkpoint.get_state(config)
print(f"\n--- Saved State ---\n{saved_state}")

# If you invoke it again with the same thread_id, it will RESUME from the last state
print("\n--- Resuming ---")
resumed_run = app_with_checkpoint.invoke(None, config) # Pass None as input to resume
print(f"\n--- Final State after Resume ---\n{resumed_run}")

## 🏁 Summary

You are now equipped with the tools to build robust, long-running, and efficient agents.

1.  **Always Design Safe Exits for Loops:** Use counters, quality checks, or other state-based conditions to prevent infinite loops.
2.  **Use `asyncio` for Parallelism:** While LangGraph doesn't have an explicit `parallel` construct, you can achieve it by using `asyncio.gather` inside your async nodes.
3.  **Checkpoints are Essential for Resilience:** Use a `checkpointer` to save and restore graph state, making your agents fault-tolerant.

Next, we'll explore how to integrate tools and external APIs into our workflows.